# Biogeophysical and hydrological processes behind the forcing, per contrast category

Surface energy, snow/albedo and water-cycle responses that sit under the Ghan terms (`radiative_forcing.ipynb`).
Plotting code: `boreal_forest_expansion.plotting.forcing_maps`. Maps: mean over 2001–2014, hatched = NOT significant (paired yearly t-test, FDR α = 0.10), each panel its own scale.

**Contrasts (user 2026-11-08), one folder per category in `results/figures/<category>/`:**

| Category | Contrasts (case1 − case2) |
|---|---|
| interactive | LCC_FUT − CTRL_PD · [LCC − CTRL]_PD · [LCC − CTRL]_FUT · CTRL_[FUT − PD] · LCC_[FUT − PD] |
| fBVOC | LCC_PD_fBVOC [lccPD − ctrlPD] · LCC_FUT_fBVOC [lccFUT − ctrlFUT] · [lccFUT − lccPD] · [lccFUT − ctrlPD] · [ctrlFUT − ctrlPD] · [lccPD − ctrlPD] (not really physical) |
| non-BVOC | [LCC_PD − CTRL_PD]_fBVOCctrlPD · [LCC_FUT − CTRL_FUT]_fBVOCctrlFUT · [LCC_FUT − LCC_PD]_fBVOCctrlPD · [LCC_FUT − CTRL_PD]_fBVOCctrlPD |
| artefacts | art_LCC_PD = LCC_PD − LCC_PD_fBVOCctrlPD · art_LCC_FUT = LCC_FUT − LCC_FUT_fBVOCctrlFUT · art_CTRL_PD = CTRL_PD − CTRL_PD_fBVOCctrlPD · art_CTRL_FUT = CTRL_FUT − CTRL_FUT_fBVOCctrlFUT |

Definitions: `forcing_maps.CONTRASTS`. **`LCC_FUT_fBVOClccFUT` is a design-B stand-in** until the run (24g) is postprocessed: LCC_FUT − art_CTRL_FUT = LCC_FUT − CTRL_FUT + CTRL_FUT_fBVOCctrlFUT (assumes art(LCC land) = art(CTRL land)); rows using it say "(design B)". **To be replaced:** set `fm.LCC_FUT_FBVOCLCCFUT_DESIGN_B = False` and rerun.

Reorganised 2026-11-08 (user); earlier figures in `results/figures-pre-reorg-20261108/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # repo src/, no pip install needed

import matplotlib.pyplot as plt
from boreal_forest_expansion.plotting.style import set_default_style
from boreal_forest_expansion.plotting import forcing_maps as fm
from boreal_forest_expansion.plotting import palette as pal

set_default_style()
START_YEAR, END_YEAR = 2001, 2014
print("Figures ->", {c: str(p.relative_to(fm.REPO_DIR)) for c, p in fm.FIG_CATEGORY.items()})
if fm.LCC_FUT_FBVOCLCCFUT_DESIGN_B:
    print("LCC_FUT_fBVOClccFUT = DESIGN-B STAND-IN (LCC_FUT − CTRL_FUT + CTRL_FUT_fBVOCctrlFUT), to be replaced by the real run")
for cat, rows in fm.CONTRASTS.items():
    print(f"{cat}:", " · ".join(c.one_line for c in rows))

for alias in fm.category_aliases():
    path = fm.filename(alias, "HYDRO", START_YEAR, END_YEAR, "lnd")
    print(f"{alias:24s}", "✓" if path.exists() else "MISSING")

## 1. Hydrology and surface fluxes
FSDS, precipitation (CAM PRECC + PRECL), ET, total runoff, snowmelt, total water storage, top-10 cm soil water, latent and sensible heat, low cloud.

In [ ]:
for cat in fm.CATEGORIES:
    fig, axes = fm.make_contrast_grid_figure(cat, fm.HYDRO_PANELS, "hydro", start_year=START_YEAR, end_year=END_YEAR)
    plt.show(); plt.close(fig)

## 2. Snow and surface albedo
Snow cover (annual and MAM), snow water equivalent, snow persistence (days), canopy snow, MAM surface albedo (CLM FSR / FSDS, ratio of month-weighted means), LAI.
The MAM albedo is the spring snow-masking signal behind the SW clear-clean term.

In [ ]:
for cat in fm.CATEGORIES:
    fig, axes = fm.make_contrast_grid_figure(cat, fm.SNOW_ALBEDO_PANELS, "snow_albedo", start_year=START_YEAR, end_year=END_YEAR)
    plt.show(); plt.close(fig)

## 3. Surface energy partition
Net surface SW, downwelling LW, surface and 2 m temperature, latent and sensible heat, Bowen ratio (SHFLX / LHFLX as a ratio of annual means, masked where |LHFLX| < 5 W m⁻²).
FLDS and TS link to the LW clear-clean term (warmer, moister boundary layer).

In [ ]:
for cat in fm.CATEGORIES:
    fig, axes = fm.make_contrast_grid_figure(cat, fm.SURFACE_ENERGY_PANELS, "surface_energy", start_year=START_YEAR, end_year=END_YEAR)
    plt.show(); plt.close(fig)

## 4. FSDS split
All-sky FSDS, clear-sky FSDSC, cloud part FSDS − FSDSC, aerosol part FSDS − FSDS_DRF, clear-clean FSDSCDRF.
These are diagnostic differences from the extra radiation calls, **not** an exact additive decomposition of FSDS.

In [ ]:
for cat in fm.CATEGORIES:
    fig, axes = fm.make_contrast_grid_figure(cat, fm.FSDS_SPLIT_PANELS, "fsds_split", start_year=START_YEAR, end_year=END_YEAR)
    plt.show(); plt.close(fig)

## 5. Land water budget: P − ET − R_liq − R_ice − ΔTWS
All terms from CLM (same land basis), mm yr⁻¹: P = RAIN + SNOW, ET = QFLX_EVAP_TOT, R_liq = QRUNOFF_TO_COUPLER, R_ice = QRUNOFF_ICE_TO_COUPLER (solid runoff from snow capping, mostly glaciers/Greenland), ΔTWS = TWS(Dec y) − TWS(Dec y−1).
ΔTWS needs the previous December, so the budget covers **2002–2014 (13 paired years)**.
Regions: LCC-modified cells (`diagnostics/lcc-modified-cells/modified_cells.nc`) and unmodified land ≥45°N; weights = area × landfrac.
The residual of each run should be ≈ 0 (budget check).

Why the coupler runoff (2026-10-25): with liquid `QRUNOFF` only, unmodified land ≥45°N left +27.7 mm yr⁻¹ unclosed in CTRL_PD (≈5 % of P). Adding the ice runoff and the land-use corrections closes every run and region to < 1 mm yr⁻¹ (CTRL_PD: +0.10 modified, −0.52 unmodified; LCC_FUT: −0.24 / −0.65).

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_rows", 300)

SETS = {cat: [c.pair for c in rows] for cat, rows in fm.CONTRASTS.items()}
LABEL = {f"{c.case1} − {c.case2}": c.one_line for rows in fm.CONTRASTS.values() for c in rows}
table = fm.water_budget_table(START_YEAR, END_YEAR, sets=SETS)
table["contrast"] = table["contrast"].map(LABEL)
out = fm.TABLE_DIR / f"water_budget_categories_{START_YEAR}_{END_YEAR}.csv"
table.to_csv(out, index=False)
print("Saved", out)
table.round(2)

In [ ]:
# Residual of each run on its own (budget check), mm/yr
table[table.term == "residual"][["contrast_set", "contrast", "region", "case1", "case2"]].round(2)

In [ ]:
for cat in fm.CATEGORIES:
    fig, axes = fm.make_water_budget_figure(table, cat, figdir=fm.FIG_CATEGORY[cat], start_year=START_YEAR, end_year=END_YEAR)
    plt.show(); plt.close(fig)